In [24]:
# 
# Importar librerías 

import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [25]:
# Leer datos
file_path = "C:/Git/vehicles_env/datasets/auto_cons_us.csv"
cars = pd.read_csv(file_path) # Cargar dataframe
print(cars.shape) # Mostrar el tamaño del dataframe

(398, 8)


In [26]:
# Preparar los datos antes de ingresarlos al algoritmo

# 1.- Eliminar los datos ausentes (6 registros) (Se tomó esa decisión porque solo representa el 1.5% del total)
cars.dropna(inplace=True)

# Esto crea una columna adicional al dataframe por cada valor diferente en la columna "Origin"
cars = pd.get_dummies(cars)

# Ya que tienes una sola característica categórica que toma tres valores unívocos diferentes, 
# nada te prohíbe utilizar get_dummies(). Tu DataFrame consigue solo dos características: 
# tres nuevas que reemplazan a una anterior. Y no introdujiste ese problemático tipo de relación 
# mayor/menor para la función, lo que no habría sido natural 
# (eso es lo que habría sucedido si hubieras usado la codificación de etiquetas).

In [27]:
# Divide los datos en características (la matriz X) y una variable objetivo (y)
X = cars.drop('Fuel consumption', axis = 1)
y = cars['Fuel consumption']

# Dividir los datos de entrenamiento (train) y datos de prueba (test)
# Especificar que el tamaño de los datos para PRUEBA será el 20% de los registros
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#------------------------ ESTANDARIZAR DATOS
scaler = StandardScaler() # Definir el modelo 
X_train_st = scaler.fit_transform(X_train) # Entrenar y Obtener el conjunto de entranamiento estandarizado
X_test_st = scaler.transform(X_test) # Obtener el conjunto de prueba estandarizado

# Declarar la lista de los modelos a utilizar
models =[Lasso(), Ridge(), DecisionTreeRegressor(), RandomForestRegressor(), GradientBoostingRegressor()]

In [28]:
def mape(y_true, y_pred):
    y_error = y_true - y_pred # calcula el vector de error
    y_error_abs = [abs(i) for i in y_error] # calcula el vector de valores absolutos de errores
    perc_error_abs = y_error_abs / y_true # calcula el vector de error relativo
    return perc_error_abs.sum() / len(y_true)

In [29]:
def make_prediction(m, X_train, y_train, X_test, y_test):
    model = m
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print('MAE:{:.2f} MSE:{:.2f} MAPE:{:.2f} R2:{:.2f} '.format(mean_absolute_error(y_test, y_pred), 
                                          mean_squared_error(y_test, y_pred),
                                                                    mape(y_test, y_pred),
                                                                    r2_score(y_test, y_pred)))

In [32]:

for i in models:
    print("\n",i)    
    make_prediction(m = i, X_train = X_train_st, y_train = y_train, X_test = X_test_st, y_test = y_test)


 Lasso()
MAE:1.56 MSE:3.77 MAPE:0.16 R2:0.81 

 Ridge()
MAE:1.02 MSE:1.78 MAPE:0.10 R2:0.91 

 DecisionTreeRegressor()
MAE:1.28 MSE:3.46 MAPE:0.11 R2:0.82 

 RandomForestRegressor()
MAE:0.89 MSE:1.68 MAPE:0.08 R2:0.91 

 GradientBoostingRegressor()
MAE:0.94 MSE:1.95 MAPE:0.09 R2:0.90 


¡Es correcto!

Los ensambles producen los mejores resultados, aunque los modelos lineales y los árboles también funcionan bien. En problemas así, la potenciación de gradiente es normalmente más precisa que un bosque aleatorio.